In [1]:
import cv2
import time
import numpy as np
import os
import glob


# Ensure project root is on PYTHONPATH so we can `import vimba_centroid_lab ...`
import sys, os, pathlib
proj_root = pathlib.Path(os.getcwd()).resolve().parent  # one level up from /intrinsics
if str(proj_root) not in sys.path:
    sys.path.append(str(proj_root))




In [2]:
from queue import Queue
from vimba_centroid_lab.camera_vimba import CameraController

save_folder = 'vimba_cam_0'
os.makedirs(save_folder, exist_ok=True)

frame_q = Queue(maxsize=50)
cam = CameraController(frame_q)
time.sleep(2)
cam.start()

# Give the camera a moment to start streaming frames
time.sleep(2)

for i in range(30):
    frame = frame_q.get()  # blocking wait for the next frame
    # Vimba streams Mono8 (grayscale) frames; convert to BGR before saving
    frame_bgr = cv2.cvtColor(frame, cv2.COLOR_GRAY2BGR)
    cv2.imwrite(f"{save_folder}/image_{i}.jpg", frame_bgr)
    print(f'photo:{i}')
    time.sleep(0.5)

cam.stop()


=== VIMBA CAMERA STARTUP ===
Getting Vimba instance...
✓ Vimba instance acquired
Discovering cameras...
✓ Found 1 camera(s)
✓ Using first available camera: DEV_000A47000665
Opening camera (AccessMode.Full)...
❌ Failed to open camera with AccessMode.Full: 'Camera' object has no attribute 'open'
✓ Camera opened successfully
Attempting to adjust GigE packet size...
✓ GigE packet size adjusted
Setting pixel format to Mono8...
⚠ Method 1 failed: 'Camera.set_pixel_format' called with unexpected argument type. Argument'fmt'. Expected type: <enum 'PixelFormat'>.
⚠ Could not set pixel format to Mono8: 'Camera' object has no attribute 'PixelFormat'
⚠ Continuing anyway - some cameras might not support this
Starting streaming with buffer_count=8...
✓ Camera streaming started successfully
=== VIMBA CAMERA STARTUP COMPLETE ===
photo:0
photo:1
photo:2
photo:3
photo:4
photo:5
photo:6
photo:7
photo:8
photo:9
photo:10
photo:11
photo:12
photo:13
photo:14
photo:15
photo:16
photo:17
photo:18
photo:19
photo

In [5]:
cam_images_folder_name = 'vimba_cam_0'
cam_images_folder_name_calibrated = f'{cam_images_folder_name}_c'
# cam_images_folder_name = 'cam_0'
os.makedirs(cam_images_folder_name_calibrated, exist_ok=True)


# Defining the dimensions of checkerboard
CHECKERBOARD = (76,36)
#CHECKERBOARD = (5,6)
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)
 
# Creating vector to store vectors of 3D points for each checkerboard image
objpoints = []
# Creating vector to store vectors of 2D points for each checkerboard image
imgpoints = [] 
 
 
# Defining the world coordinates for 3D points
objp = np.zeros((1, CHECKERBOARD[0] * CHECKERBOARD[1], 3), np.float32)
objp[0,:,:2] = np.mgrid[0:CHECKERBOARD[0], 0:CHECKERBOARD[1]].T.reshape(-1, 2)
prev_img_shape = None
 
# Extracting path of individual image stored in a given directory
images = glob.glob(f'./{cam_images_folder_name}/*.jpg')
for fname in images:
    img = cv2.imread(fname)
    gray = cv2.cvtColor(img,cv2.COLOR_BGR2GRAY)
    # Find the chess board corners
    # If desired number of corners are found in the image then ret = true
    # ret, corners = cv2.findChessboardCorners(gray, CHECKERBOARD, cv2.CALIB_CB_ADAPTIVE_THRESH + cv2.CALIB_CB_FAST_CHECK + cv2.CALIB_CB_NORMALIZE_IMAGE)
    ret, corners = cv2.findChessboardCorners(gray, CHECKERBOARD, cv2.CALIB_CB_ADAPTIVE_THRESH + cv2.CALIB_CB_NORMALIZE_IMAGE)
     
    """
    If desired number of corner are detected,
    we refine the pixel coordinates and display 
    them on the images of checker board
    """
    if ret == True:
        objpoints.append(objp)
        # refining pixel coordinates for given 2d points.
        corners2 = cv2.cornerSubPix(gray, corners, (11,11),(-1,-1), criteria)
         
        imgpoints.append(corners2)
 
        # Draw and display the corners
        img = cv2.drawChessboardCorners(img, CHECKERBOARD, corners2, ret)
     
    cv2.imshow('img',img)
    # cv2.waitKey(0)
    
    new_frame_name = cam_images_folder_name_calibrated + '/' + os.path.basename(fname)
    print(new_frame_name)
    cv2.imwrite(new_frame_name, img)

 
# cv2.destroyAllWindows()
 
h,w = img.shape[:2]
 
"""
Performing camera calibration by 
passing the value of known 3D points (objpoints)
and corresponding pixel coordinates of the 
detected corners (imgpoints)
"""
ret, mtx, dist, rvecs, tvecs = cv2.calibrateCamera(objpoints, imgpoints, gray.shape[::-1], None, None)
 
print("Camera matrix : \n")
print(mtx)
print("dist : \n")
print(dist)

vimba_cam_0_c/image_9.jpg
vimba_cam_0_c/image_12.jpg
vimba_cam_0_c/image_11.jpg
vimba_cam_0_c/image_4.jpg
vimba_cam_0_c/image_18.jpg
vimba_cam_0_c/image_6.jpg
vimba_cam_0_c/image_29.jpg
vimba_cam_0_c/image_7.jpg
vimba_cam_0_c/image_23.jpg
vimba_cam_0_c/image_14.jpg
vimba_cam_0_c/image_17.jpg
vimba_cam_0_c/image_22.jpg


KeyboardInterrupt: 

In [6]:
import os, glob, cv2, numpy as np, json

# === Folder for each camera ===
camera_folders = ["vimba_cam_0", "vimba_cam_1", "vimba_cam_2"]  # add all your cam folders

CHECKERBOARD = (76, 36)   # inner corners (77x37 squares)
SQUARE_SIZE_MM = 25.0
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 100, 1e-6)

results = {}

for cam_folder in camera_folders:
    out_dir = f"{cam_folder}_c"
    os.makedirs(out_dir, exist_ok=True)

    # prepare object points
    nx, ny = CHECKERBOARD
    objp = np.zeros((nx*ny, 3), np.float32)
    objp[:, :2] = np.mgrid[0:nx, 0:ny].T.reshape(-1, 2)
    objp *= SQUARE_SIZE_MM

    objpoints, imgpoints = [], []
    images = sorted(glob.glob(f'./{cam_folder}/*.jpg'))
    if not images:
        print(f"No images found for {cam_folder}")
        continue

    for fname in images:
        img  = cv2.imread(fname)
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        ret, corners = cv2.findChessboardCorners(gray, CHECKERBOARD, None)
        if ret:
            corners2 = cv2.cornerSubPix(gray, corners, (11,11), (-1,-1), criteria)
            objpoints.append(objp)
            imgpoints.append(corners2)

    h, w = gray.shape[:2]
    ret, K, dist, rvecs, tvecs = cv2.calibrateCamera(
        objpoints, imgpoints, (w, h), None, None
    )

    # === Store only what you need ===
    results[cam_folder] = {
        "intrinsic_matrix": K.tolist(),
        "distortion_coef": dist.ravel().tolist(),
        "rotation": 0
    }

# Save everything into one JSON
with open("all_cameras_intrinsics.json", "w") as f:
    json.dump(results, f, indent=4)

print("Saved intrinsics to all_cameras_intrinsics.json")


error: OpenCV(4.12.0) /io/opencv/modules/calib3d/src/calibration.cpp:1379: error: (-215:Assertion failed) nimages > 0 in function 'calibrateCameraRO'
